In [ ]:
!pip install -q -U funasr modelscope transformers openai-whisper sentencepiece lightgbm catboost
import os
import re
import gc
import unicodedata
import random
import wave
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
import whisper
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from funasr import AutoModel as FunASRModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, log_loss, classification_report
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from scipy.optimize import minimize
import lightgbm as lgb
from catboost import CatBoostClassifier
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
def clear_gpu():
    import sys
    if hasattr(sys, 'last_traceback'):
        sys.last_traceback = None
    for obj in gc.get_objects():
        try:
            if isinstance(obj, torch.nn.Module):
                obj.cpu()
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
seed_everything(42)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")


In [ ]:
DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.csv' in files:
        DATA_DIR = Path(root)
        break
if DATA_DIR is None:
    for root, dirs, files in os.walk('.'):
        if 'train.csv' in files:
            DATA_DIR = Path(root)
            break
if DATA_DIR is None:
    raise FileNotFoundError("Could not find train.csv")
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
train_df.loc[train_df['Id'] == 636, 'emotion'] = 'neutral'
train_audio_dir = DATA_DIR / 'train'
test_audio_dir = DATA_DIR / 'test'
target_emotions = ['angry', 'happy', 'neutral', 'sad']
label2id = {label: i for i, label in enumerate(target_emotions)}
id2label = {i: label for i, label in enumerate(target_emotions)}
train_df['label'] = train_df['emotion'].map(label2id)
y_train = train_df['label'].values
def load_raw_wav(file_path):
    with wave.open(str(file_path), 'rb') as w:
        sw = w.getsampwidth()
        frames = w.readframes(w.getnframes())
        if sw == 2:
            arr = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
        else:
            arr = np.frombuffer(frames, dtype=np.int32).astype(np.float32) / 2147483648.0
    return arr, sw
def extract_signal_prosody_and_pitch(arr, sw):
    sr = 16000
    dur = len(arr) / float(sr)
    rms = float(np.sqrt(np.mean(arr**2)) + 1e-8)
    peak = float(np.max(np.abs(arr)) + 1e-8)
    crest = peak / rms
    zcr = float(np.mean(np.diff(np.sign(arr)) != 0))
    abs_arr = np.abs(arr)
    q10, q25, q50, q75, q90, q99 = np.percentile(abs_arr, [10, 25, 50, 75, 90, 99])
    win = 320
    n_f = max(len(arr) // win, 1)
    frames = arr[:n_f*win].reshape(n_f, win) if len(arr) >= win else arr.reshape(1, -1)
    f_rms = np.sqrt(np.mean(frames**2, axis=1)) + 1e-8
    f_zcr = np.mean(np.diff(np.sign(frames), axis=1) != 0, axis=1)
    sil_05 = float(np.mean(f_rms < 0.05 * np.max(f_rms)))
    sil_10 = float(np.mean(f_rms < 0.10 * np.max(f_rms)))
    sil_20 = float(np.mean(f_rms < 0.20 * np.max(f_rms)))
    rms_std = float(np.std(f_rms))
    rms_cv = rms_std / float(np.mean(f_rms))
    rms_max = float(np.max(f_rms))
    rms_p90_p10 = float(np.percentile(f_rms, 90) / (np.percentile(f_rms, 10) + 1e-6))
    zcr_std = float(np.std(f_zcr))
    p_win = 640
    n_pf = max(len(arr) // p_win, 1)
    p_frames = arr[:n_pf*p_win].reshape(n_pf, p_win) if len(arr) >= p_win else arr.reshape(1, -1)
    min_lag = sr // 400
    max_lag = min(sr // 70, p_frames.shape[1] - 1)
    f0_list = []
    if max_lag > min_lag:
        for pf in p_frames:
            pf_c = pf - np.mean(pf)
            eng = np.sum(pf_c**2)
            if eng > 1e-4:
                ac = np.correlate(pf_c, pf_c, mode='full')[len(pf_c)-1:]
                ac_slice = ac[min_lag:max_lag]
                if len(ac_slice) > 0:
                    peak_lag = np.argmax(ac_slice) + min_lag
                    if ac[peak_lag] / (eng + 1e-8) > 0.35:
                        f0_list.append(float(sr) / peak_lag)
    if len(f0_list) > 0:
        f0_arr = np.array(f0_list)
        f0_mean = float(np.mean(f0_arr)) / 200.0
        f0_std = float(np.std(f0_arr)) / 100.0
        f0_p90 = float(np.percentile(f0_arr, 90)) / 200.0
        f0_p10 = float(np.percentile(f0_arr, 10)) / 200.0
        f0_range = f0_p90 - f0_p10
        voiced_ratio = float(len(f0_arr)) / float(n_pf)
    else:
        f0_mean, f0_std, f0_p90, f0_p10, f0_range, voiced_ratio = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    fft_mag = np.abs(np.fft.rfft(arr[:min(len(arr), 64000)])) + 1e-8
    freqs = np.linspace(0, 1, len(fft_mag))
    spec_centroid = float(np.sum(freqs * fft_mag) / np.sum(fft_mag))
    spec_bw = float(np.sqrt(np.sum(((freqs - spec_centroid)**2) * fft_mag) / np.sum(fft_mag)))
    cumsum = np.cumsum(fft_mag)
    spec_rolloff = float(np.searchsorted(cumsum, 0.85 * cumsum[-1]) / len(fft_mag))
    bands = [float(np.log(np.mean(b) + 1e-8)) for b in np.array_split(fft_mag, 12)]
    stft_frames = np.abs(np.fft.rfft(frames * np.hanning(frames.shape[1]), axis=1)) + 1e-8
    band_splits = np.array_split(stft_frames, 12, axis=1)
    band_means = [float(np.mean(np.log(b))) for b in band_splits]
    band_stds = [float(np.std(np.log(b))) for b in band_splits]
    base_feats = [
        dur, np.log(dur + 1e-3), rms, np.log(rms), peak, crest, zcr, zcr_std,
        q10, q25, q50, q75, q90, q99,
        sil_05, sil_10, sil_20, rms_std, rms_cv, rms_max, rms_p90_p10,
        f0_mean, f0_std, f0_p90, f0_p10, f0_range, voiced_ratio,
        spec_centroid, spec_bw, spec_rolloff, float(sw)
    ]
    return np.array(base_feats + bands + band_means + band_stds, dtype=np.float32)
train_wavs, train_prosody = [], []
for f in tqdm(train_df['filename'], desc="Loading train waveforms & prosody"):
    arr, sw = load_raw_wav(train_audio_dir / f)
    train_wavs.append(arr)
    train_prosody.append(extract_signal_prosody_and_pitch(arr, sw))
test_wavs, test_prosody = [], []
for f in tqdm(test_df['filename'], desc="Loading test waveforms & prosody"):
    arr, sw = load_raw_wav(test_audio_dir / f)
    test_wavs.append(arr)
    test_prosody.append(extract_signal_prosody_and_pitch(arr, sw))
X_train_prosody = np.vstack(train_prosody)
X_test_prosody = np.vstack(test_prosody)
print(f"Prosody & Pitch features shape: {X_train_prosody.shape}")


In [ ]:
print("--- 1. Extracting emotion2vec+ Large Embeddings & Full 9-Class Emotion Priors ---")
e2v_model = FunASRModel(
    model="iic/emotion2vec_plus_large",
    hub="modelscope",
    device=device
)
all_9_emotions = ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'other', 'sad', 'surprised', 'unknown']
def extract_acoustic_data(df, audio_dir):
    embeddings = []
    priors_4 = []
    priors_9 = []
    for filename in tqdm(df['filename'], desc="emotion2vec+"):
        audio_path = str(audio_dir / filename)
        res = e2v_model.generate(input=audio_path, output_dir=None, granularity="utterance", extract_embedding=True)
        feats = res[0]["feats"]
        if isinstance(feats, torch.Tensor):
            feats = feats.detach().cpu().numpy()
        feats = np.array(feats).squeeze()
        embeddings.append(feats)
        labels_out = [str(l).lower() for l in res[0]["labels"]]
        scores_out = res[0]["scores"]
        score_dict = {lbl.split('/')[-1]: sc for lbl, sc in zip(labels_out, scores_out)}
        p4 = [score_dict.get(emo, 1e-5) for emo in target_emotions]
        s4 = sum(p4)
        priors_4.append([p / s4 for p in p4])
        p9 = [score_dict.get(emo, 1e-5) for emo in all_9_emotions]
        s9 = sum(p9)
        priors_9.append([p / s9 for p in p9])
    return np.vstack(embeddings), np.array(priors_4, dtype=np.float32), np.array(priors_9, dtype=np.float32)
X_train_audio, train_priors4, train_priors9 = extract_acoustic_data(train_df, train_audio_dir)
X_test_audio, test_priors4, test_priors9 = extract_acoustic_data(test_df, test_audio_dir)
del e2v_model
clear_gpu()
print("--- 2. Extracting Microsoft WavLM-Base-Plus Multi-Layer Frame Mean+Std ---")
wavlm_model = AutoModel.from_pretrained("microsoft/wavlm-base-plus", output_hidden_states=True).to(device)
wavlm_model.eval()
def extract_wavlm_features(wav_list):
    embs = []
    for arr in tqdm(wav_list, desc="WavLM-Base-Plus"):
        w = arr[:128000]
        if len(w) < 1600:
            w = np.pad(w, (0, 1600 - len(w)))
        w_norm = (w - np.mean(w)) / (np.std(w) + 1e-7)
        inp = torch.tensor(w_norm, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            out = wavlm_model(inp)
            h = (out.hidden_states[-1] + out.hidden_states[-2] + out.hidden_states[-3]) / 3.0
            m = h.mean(dim=1).squeeze(0).cpu().numpy()
            s = h.std(dim=1).squeeze(0).cpu().numpy()
            embs.append(np.concatenate([m, s]))
    return np.vstack(embs)
X_tr_wavlm = extract_wavlm_features(train_wavs)
X_te_wavlm = extract_wavlm_features(test_wavs)
wavlm_model.cpu()
del wavlm_model
clear_gpu()
print(f"e2v shape: {X_train_audio.shape}, WavLM shape: {X_tr_wavlm.shape}")


In [ ]:
clear_gpu()
print("--- High-Precision Hindi ASR with Whisper Large-v3 (Beam Search) ---")
whisper_model = whisper.load_model("large-v3", device=device)
def transcribe_audios(df, audio_dir):
    texts = []
    asr_meta = []
    for filename in tqdm(df['filename'], desc="Whisper Large-v3 Beam=5"):
        audio_path = str(audio_dir / filename)
        res = whisper_model.transcribe(
            audio_path,
            language="hi",
            task="transcribe",
            fp16=torch.cuda.is_available(),
            temperature=0.0,
            beam_size=5,
            best_of=5,
            condition_on_previous_text=False
        )
        t = res["text"].strip()
        texts.append(t if t else "खाली")
        segs = res.get("segments", [])
        if len(segs) > 0:
            avg_lp = float(np.mean([s.get("avg_logprob", -0.5) for s in segs]))
            no_sp = float(np.mean([s.get("no_speech_prob", 0.1) for s in segs]))
        else:
            avg_lp, no_sp = -1.0, 0.5
        asr_meta.append([avg_lp, no_sp, float(len(t)), float(len(t.split()))])
    return texts, np.array(asr_meta, dtype=np.float32)
train_texts, X_tr_asrmeta = transcribe_audios(train_df, train_audio_dir)
test_texts, X_te_asrmeta = transcribe_audios(test_df, test_audio_dir)
whisper_model.cpu()
del whisper_model
clear_gpu()
def normalize_hindi_text(text):
    t = unicodedata.normalize('NFC', str(text)).lower()
    t = t.replace('ँ', 'ं')
    t = t.replace('़', '')
    for ch in '।|?!,.-"()[]{}:;':
        t = t.replace(ch, ' ')
    t = ' '.join(t.split())
    return t if t else "खाली"
train_texts_norm = [normalize_hindi_text(t) for t in train_texts]
test_texts_norm = [normalize_hindi_text(t) for t in test_texts]
def extract_multilayer_mean_max(model_name, tr_texts, te_texts):
    print(f"Extracting 3-layer Mean+Max text features from: {model_name}")
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name, output_hidden_states=True).to(device)
    mdl.eval()
    def _run(texts):
        embs = []
        bs = 32
        for i in range(0, len(texts), bs):
            batch = texts[i:i+bs]
            inp = tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            with torch.no_grad():
                out = mdl(**inp)
                hs = out.hidden_states
                hidden = (hs[-1] + hs[-2] + hs[-3]) / 3.0
                mask = inp['attention_mask'].unsqueeze(-1).expand(hidden.size()).float()
                mean_pool = torch.sum(hidden * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
                hidden_masked = hidden.masked_fill(mask == 0, -1e4)
                max_pool = torch.max(hidden_masked, 1)[0]
                pooled = 0.70 * mean_pool + 0.30 * max_pool
                embs.append(pooled.cpu().numpy())
        return np.vstack(embs)
    tr_emb = _run(tr_texts)
    te_emb = _run(te_texts)
    mdl.cpu()
    del mdl
    clear_gpu()
    return tr_emb, te_emb
X_tr_hroberta, X_te_hroberta = extract_multilayer_mean_max("l3cube-pune/hindi-roberta", train_texts, test_texts)
X_tr_muril, X_te_muril = extract_multilayer_mean_max("google/muril-base-cased", train_texts, test_texts)
X_tr_xlmr, X_te_xlmr = extract_multilayer_mean_max("xlm-roberta-large", train_texts, test_texts)
print("Extracting Multilingual Sentiment Embeddings & Valence Logits...")
sent_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
sent_tok = AutoTokenizer.from_pretrained(sent_name)
sent_mdl = AutoModelForSequenceClassification.from_pretrained(sent_name, output_hidden_states=True).to(device)
sent_mdl.eval()
def extract_sentiment_features(texts):
    embs, logits_list = [], []
    bs = 32
    for i in range(0, len(texts), bs):
        batch = texts[i:i+bs]
        inp = sent_tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            out = sent_mdl(**inp)
            logits_list.append(out.logits.cpu().numpy())
            hidden = (out.hidden_states[-1] + out.hidden_states[-2]) * 0.5
            mask = inp['attention_mask'].unsqueeze(-1).expand(hidden.size()).float()
            mean_pool = torch.sum(hidden * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
            hidden_masked = hidden.masked_fill(mask == 0, -1e4)
            max_pool = torch.max(hidden_masked, 1)[0]
            pooled = 0.70 * mean_pool + 0.30 * max_pool
            embs.append(pooled.cpu().numpy())
    return np.vstack(embs), np.vstack(logits_list)
X_tr_sent_emb, X_tr_sent_logits = extract_sentiment_features(train_texts)
X_te_sent_emb, X_te_sent_logits = extract_sentiment_features(test_texts)
sent_mdl.cpu()
del sent_mdl
clear_gpu()
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, max_features=7000, sublinear_tf=True)
tfidf_word = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=2, max_features=5000, sublinear_tf=True)
X_tr_tfidf = np.hstack([
    tfidf_char.fit_transform(train_texts_norm).toarray(),
    tfidf_word.fit_transform(train_texts_norm).toarray()
])
X_te_tfidf = np.hstack([
    tfidf_char.transform(test_texts_norm).toarray(),
    tfidf_word.transform(test_texts_norm).toarray()
])
print(f"Sentiment logits: {X_tr_sent_logits.shape}, TF-IDF shape: {X_tr_tfidf.shape}")


In [ ]:
class TextClsDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
def finetune_text_backbone_cv(model_name, tr_texts, y_all, te_texts, skf_splits, epochs=4, lr=2.5e-5, seed_base=42):
    print(f"--- 5-Fold Supervised Fine-Tuning: {model_name} ---")
    tok = AutoTokenizer.from_pretrained(model_name)
    tr_enc = tok(tr_texts, padding=True, truncation=True, max_length=96, return_tensors="pt")
    te_enc = tok(te_texts, padding=True, truncation=True, max_length=96, return_tensors="pt")
    oof_p = np.zeros((len(tr_texts), 4), dtype=np.float32)
    te_p = np.zeros((len(te_texts), 4), dtype=np.float32)
    te_ds = TextClsDataset(te_enc)
    te_loader = DataLoader(te_ds, batch_size=32, shuffle=False)
    for fold, (train_idx, val_idx) in enumerate(skf_splits):
        seed_everything(seed_base + fold)
        mdl = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=4,
            ignore_mismatched_sizes=True
        ).to(device)
        sub_tr_enc = {k: v[train_idx] for k, v in tr_enc.items()}
        sub_va_enc = {k: v[val_idx] for k, v in tr_enc.items()}
        tr_loader = DataLoader(TextClsDataset(sub_tr_enc, y_all[train_idx]), batch_size=16, shuffle=True)
        va_loader = DataLoader(TextClsDataset(sub_va_enc), batch_size=32, shuffle=False)
        opt = torch.optim.AdamW(mdl.parameters(), lr=lr, weight_decay=0.02)
        total_steps = len(tr_loader) * epochs
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps)
        crit = nn.CrossEntropyLoss(label_smoothing=0.05)
        scaler_amp = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
        best_va_f1 = -1.0
        best_va_probs = None
        best_te_probs = None
        for ep in range(epochs):
            mdl.train()
            for batch in tr_loader:
                opt.zero_grad()
                inp_ids = batch['input_ids'].to(device)
                att_mask = batch['attention_mask'].to(device)
                lbls = batch['labels'].to(device)
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    out = mdl(input_ids=inp_ids, attention_mask=att_mask)
                    loss = crit(out.logits, lbls)
                scaler_amp.scale(loss).backward()
                scaler_amp.step(opt)
                scaler_amp.update()
                sched.step()
            mdl.eval()
            va_preds_list = []
            with torch.no_grad():
                for batch in va_loader:
                    inp_ids = batch['input_ids'].to(device)
                    att_mask = batch['attention_mask'].to(device)
                    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                        out = mdl(input_ids=inp_ids, attention_mask=att_mask)
                        p = F.softmax(out.logits.float(), dim=-1).cpu().numpy()
                    va_preds_list.append(p)
            va_p = np.vstack(va_preds_list)
            f1 = f1_score(y_all[val_idx], np.argmax(va_p, axis=1), average='macro')
            if f1 >= best_va_f1:
                best_va_f1 = f1
                best_va_probs = va_p
                te_preds_list = []
                with torch.no_grad():
                    for batch in te_loader:
                        inp_ids = batch['input_ids'].to(device)
                        att_mask = batch['attention_mask'].to(device)
                        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                            out = mdl(input_ids=inp_ids, attention_mask=att_mask)
                            p = F.softmax(out.logits.float(), dim=-1).cpu().numpy()
                        te_preds_list.append(p)
                best_te_probs = np.vstack(te_preds_list)
        oof_p[val_idx] = best_va_probs
        te_p += best_te_probs / len(skf_splits)
        mdl.cpu()
        del mdl, opt, sched
        clear_gpu()
    print(f"Fine-Tuned {model_name} Text-Only OOF Macro F1: {f1_score(y_all, np.argmax(oof_p, axis=1), average='macro'):.4f}")
    return oof_p, te_p
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_splits = list(skf.split(train_df, y_train))
oof_ft_hrob, te_ft_hrob = finetune_text_backbone_cv("l3cube-pune/hindi-roberta", train_texts, y_train, test_texts, skf_splits, epochs=4, lr=2.5e-5, seed_base=42)
oof_ft_sent, te_ft_sent = finetune_text_backbone_cv("cardiffnlp/twitter-xlm-roberta-base-sentiment", train_texts, y_train, test_texts, skf_splits, epochs=4, lr=2.5e-5, seed_base=142)


In [ ]:
def scale_and_block_norm(tr_mat, te_mat, weight=1.0):
    sc = StandardScaler()
    tr_s = sc.fit_transform(tr_mat)
    te_s = sc.transform(te_mat)
    d_sqrt = np.sqrt(tr_mat.shape[1])
    return tr_s, te_s, (tr_s / d_sqrt) * weight, (te_s / d_sqrt) * weight
X_tr_a1, X_te_a1, B_tr_a1, B_te_a1 = scale_and_block_norm(X_train_audio, X_test_audio, weight=1.35)
X_tr_a2, X_te_a2, B_tr_a2, B_te_a2 = scale_and_block_norm(X_tr_wavlm, X_te_wavlm, weight=0.95)
X_tr_p, X_te_p, B_tr_p, B_te_p = scale_and_block_norm( np.hstack([X_train_prosody, X_tr_asrmeta]), np.hstack([X_test_prosody, X_te_asrmeta]), weight=0.75)
ft_tr_log = np.hstack([np.log(np.clip(oof_ft_hrob, 1e-4, 1.0)), np.log(np.clip(oof_ft_sent, 1e-4, 1.0))])
ft_te_log = np.hstack([np.log(np.clip(te_ft_hrob, 1e-4, 1.0)), np.log(np.clip(te_ft_sent, 1e-4, 1.0))])
tr_priors_raw = np.hstack([np.log(np.clip(train_priors4, 1e-5, 1.0)), np.log(np.clip(train_priors9, 1e-5, 1.0)), X_tr_sent_logits, ft_tr_log])
te_priors_raw = np.hstack([np.log(np.clip(test_priors4, 1e-5, 1.0)), np.log(np.clip(test_priors9, 1e-5, 1.0)), X_te_sent_logits, ft_te_log])
X_tr_pr, X_te_pr, B_tr_pr, B_te_pr = scale_and_block_norm(tr_priors_raw, te_priors_raw, weight=1.10)
X_tr_t1, X_te_t1, B_tr_t1, B_te_t1 = scale_and_block_norm(X_tr_hroberta, X_te_hroberta, weight=1.15)
X_tr_t2, X_te_t2, B_tr_t2, B_te_t2 = scale_and_block_norm(X_tr_muril, X_te_muril, weight=1.00)
X_tr_t3, X_te_t3, B_tr_t3, B_te_t3 = scale_and_block_norm(X_tr_xlmr, X_te_xlmr, weight=1.20)
X_tr_t4, X_te_t4, B_tr_t4, B_te_t4 = scale_and_block_norm(X_tr_sent_emb, X_te_sent_emb, weight=1.05)
B_tr_tfidf = normalize(X_tr_tfidf, axis=1) * 1.15
B_te_tfidf = normalize(X_te_tfidf, axis=1) * 1.15
Xa_train_full = np.hstack([X_tr_a1, X_tr_a2, X_tr_p, X_tr_pr])
Xa_test_full = np.hstack([X_te_a1, X_te_a2, X_te_p, X_te_pr])
Xt_train_full = np.hstack([X_tr_t1, X_tr_t2, X_tr_t3, X_tr_t4])
Xt_test_full = np.hstack([X_te_t1, X_te_t2, X_te_t3, X_te_t4])
X_tr_all = np.hstack([Xa_train_full, Xt_train_full])
X_te_all = np.hstack([Xa_test_full, Xt_test_full])
B_tr_all = np.hstack([B_tr_a1, B_tr_a2, B_tr_p, B_tr_pr, B_tr_t1, B_tr_t2, B_tr_t3, B_tr_t4, B_tr_tfidf])
B_te_all = np.hstack([B_te_a1, B_te_a2, B_te_p, B_te_pr, B_te_t1, B_te_t2, B_te_t3, B_te_t4, B_te_tfidf])
test_by_len = defaultdict(list)
for idx, w in enumerate(test_wavs):
    test_by_len[len(w)].append(idx)
test_twin_groups = []
visited_te = set()
for i in range(len(test_wavs)):
    if i in visited_te:
        continue
    grp = [i]
    visited_te.add(i)
    for k in test_by_len[len(test_wavs[i])]:
        if k not in visited_te:
            if np.corrcoef(test_wavs[i], test_wavs[k])[0, 1] > 0.90:
                grp.append(k)
                visited_te.add(k)
    if len(grp) > 1:
        test_twin_groups.append(grp)
        X_te_all[grp] = np.mean(X_te_all[grp], axis=0, keepdims=True)
        B_te_all[grp] = np.mean(B_te_all[grp], axis=0, keepdims=True)
        Xa_test_full[grp] = np.mean(Xa_test_full[grp], axis=0, keepdims=True)
        Xt_test_full[grp] = np.mean(Xt_test_full[grp], axis=0, keepdims=True)
        X_te_tfidf[grp] = np.mean(X_te_tfidf[grp], axis=0, keepdims=True)
        te_ft_hrob[grp] = np.mean(te_ft_hrob[grp], axis=0, keepdims=True)
        te_ft_sent[grp] = np.mean(te_ft_sent[grp], axis=0, keepdims=True)
class GatedBilinearMultimodalNet(nn.Module):
    def __init__(self, d_audio, d_text, n_classes=4):
        super().__init__()
        self.a_net = nn.Sequential(
            nn.Dropout(0.35),
            nn.Linear(d_audio, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.25)
        )
        self.t_net = nn.Sequential(
            nn.Dropout(0.35),
            nn.Linear(d_text, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.25)
        )
        self.gate = nn.Sequential(
            nn.Linear(512, 256),
            nn.Sigmoid()
        )
        self.head = nn.Sequential(
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, n_classes)
        )
    def forward(self, xa, xt):
        ha = self.a_net(xa)
        ht = self.t_net(xt)
        g = self.gate(torch.cat([ha, ht], dim=1))
        gated_mix = g * ha + (1.0 - g) * ht
        bilinear = ha * ht
        fused = torch.cat([ha, ht, gated_mix, bilinear], dim=1)
        return self.head(fused)
def train_neural_fold(Xa_tr, Xt_tr, y_tr, Xa_va, Xt_va, y_va, Xa_te, Xt_te, seed):
    seed_everything(seed)
    model = GatedBilinearMultimodalNet(Xa_tr.shape[1], Xt_tr.shape[1], 4).to(device)
    ds = TensorDataset(torch.tensor(Xa_tr, dtype=torch.float32), torch.tensor(Xt_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long))
    loader = DataLoader(ds, batch_size=32, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=2e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=28)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_f1 = -1.0
    best_va_probs = None
    best_te_probs = None
    xa_va_t = torch.tensor(Xa_va, dtype=torch.float32).to(device)
    xt_va_t = torch.tensor(Xt_va, dtype=torch.float32).to(device)
    xa_te_t = torch.tensor(Xa_te, dtype=torch.float32).to(device)
    xt_te_t = torch.tensor(Xt_te, dtype=torch.float32).to(device)
    for epoch in range(28):
        model.train()
        for ba, bt, by in loader:
            ba, bt, by = ba.to(device), bt.to(device), by.to(device)
            opt.zero_grad()
            if random.random() < 0.35:
                lam = float(np.random.beta(0.3, 0.3))
                idx_perm = torch.randperm(ba.size(0)).to(device)
                logits = model(lam * ba + (1.0 - lam) * ba[idx_perm], lam * bt + (1.0 - lam) * bt[idx_perm])
                loss = lam * crit(logits, by) + (1.0 - lam) * crit(logits, by[idx_perm])
            else:
                loss = crit(model(ba, bt), by)
            loss.backward()
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            va_logits = model(xa_va_t, xt_va_t)
            va_p = F.softmax(va_logits, dim=-1).cpu().numpy()
            f1 = f1_score(y_va, np.argmax(va_p, axis=1), average='macro')
            if f1 >= best_f1:
                best_f1 = f1
                best_va_probs = va_p
                te_logits = model(xa_te_t, xt_te_t)
                best_te_probs = F.softmax(te_logits, dim=-1).cpu().numpy()
    return best_va_probs, best_te_probs
def cosine_knn_predict(X_tr, y_tr, X_query, k=9, temp=0.07):
    X_tr_n = normalize(X_tr, axis=1)
    X_q_n = normalize(X_query, axis=1)
    sim = np.dot(X_q_n, X_tr_n.T)
    probs = np.zeros((len(X_query), 4), dtype=np.float32)
    for i in range(len(X_query)):
        top_idx = np.argsort(sim[i])[-k:]
        top_sim = sim[i, top_idx]
        weights = np.exp((top_sim - np.max(top_sim)) / temp)
        for idx_n, w in zip(top_idx, weights):
            probs[i, y_tr[idx_n]] += w
        probs[i] /= np.sum(probs[i]) + 1e-8
    return probs
def find_exact_waveform_twins(query_wavs, ref_wavs, ref_labels, exclude_self=False):
    ref_by_len = defaultdict(list)
    for j, rw in enumerate(ref_wavs):
        ref_by_len[len(rw)].append(j)
    twin_probs = np.zeros((len(query_wavs), 4), dtype=np.float32)
    has_twin = np.zeros(len(query_wavs), dtype=bool)
    for i, qw in enumerate(query_wavs):
        votes = np.zeros(4, dtype=np.float32)
        for j in ref_by_len.get(len(qw), []):
            if exclude_self and i == j:
                continue
            c = np.corrcoef(qw, ref_wavs[j])[0, 1]
            if c > 0.90:
                votes[ref_labels[j]] += c ** 4
        if np.sum(votes) > 0:
            has_twin[i] = True
            best_label = int(np.argmax(votes))
            p = np.full(4, 0.004 / 3.0, dtype=np.float32)
            p[best_label] = 0.996
            twin_probs[i] = p
    return has_twin, twin_probs
def compute_class_prototype_sims(X_tr, y_tr, X_va, X_te):
    X_tr_n = normalize(X_tr, axis=1)
    X_va_n = normalize(X_va, axis=1)
    X_te_n = normalize(X_te, axis=1)
    centroids = np.vstack([normalize(np.mean(X_tr_n[y_tr == c], axis=0, keepdims=True)) for c in range(4)])
    return np.dot(X_va_n, centroids.T), np.dot(X_te_n, centroids.T)
n_splits = 5
oof_probs = np.zeros((len(train_df), 4))
test_probs = np.zeros((len(test_df), 4))
for fold, (train_idx, val_idx) in enumerate(skf_splits):
    y_tr, y_va = y_train[train_idx], y_train[val_idx]
    lr_bn = LogisticRegression(C=0.45, max_iter=2500, random_state=42 + fold)
    lr_bn.fit(B_tr_all[train_idx], y_tr)
    p_lrbn_va = lr_bn.predict_proba(B_tr_all[val_idx])
    p_lrbn_te = lr_bn.predict_proba(B_te_all)
    svc_bn = SVC(C=2.4, kernel='rbf', probability=True, random_state=42 + fold)
    svc_bn.fit(B_tr_all[train_idx], y_tr)
    p_svcbn_va = svc_bn.predict_proba(B_tr_all[val_idx])
    p_svcbn_te = svc_bn.predict_proba(B_te_all)
    svc_std = SVC(C=2.2, kernel='rbf', probability=True, random_state=42 + fold)
    svc_std.fit(X_tr_all[train_idx], y_tr)
    p_svcstd_va = svc_std.predict_proba(X_tr_all[val_idx])
    p_svcstd_te = svc_std.predict_proba(X_te_all)
    lr_std = LogisticRegression(C=0.08, max_iter=2500, random_state=42 + fold)
    lr_std.fit(X_tr_all[train_idx], y_tr)
    p_lrstd_va = lr_std.predict_proba(X_tr_all[val_idx])
    p_lrstd_te = lr_std.predict_proba(X_te_all)
    mlp_bn = MLPClassifier(hidden_layer_sizes=(512, 128), activation='relu', alpha=0.12, max_iter=600, random_state=42 + fold)
    mlp_bn.fit(B_tr_all[train_idx], y_tr)
    p_mlp_va = mlp_bn.predict_proba(B_tr_all[val_idx])
    p_mlp_te = mlp_bn.predict_proba(B_te_all)
    p_gmu1_va, p_gmu1_te = train_neural_fold(
        Xa_train_full[train_idx], Xt_train_full[train_idx], y_tr,
        Xa_train_full[val_idx], Xt_train_full[val_idx], y_va,
        Xa_test_full, Xt_test_full, seed=100 + fold
    )
    p_gmu2_va, p_gmu2_te = train_neural_fold(
        Xa_train_full[train_idx], Xt_train_full[train_idx], y_tr,
        Xa_train_full[val_idx], Xt_train_full[val_idx], y_va,
        Xa_test_full, Xt_test_full, seed=300 + fold
    )
    p_gmu3_va, p_gmu3_te = train_neural_fold(
        Xa_train_full[train_idx], Xt_train_full[train_idx], y_tr,
        Xa_train_full[val_idx], Xt_train_full[val_idx], y_va,
        Xa_test_full, Xt_test_full, seed=500 + fold
    )
    p_gmu_va = (p_gmu1_va + p_gmu2_va + p_gmu3_va) / 3.0
    p_gmu_te = (p_gmu1_te + p_gmu2_te + p_gmu3_te) / 3.0
    X_txt_lex_tr = np.hstack([B_tr_t1[train_idx], B_tr_t2[train_idx], B_tr_t3[train_idx], B_tr_t4[train_idx], B_tr_tfidf[train_idx] * 1.4])
    X_txt_lex_va = np.hstack([B_tr_t1[val_idx], B_tr_t2[val_idx], B_tr_t3[val_idx], B_tr_t4[val_idx], B_tr_tfidf[val_idx] * 1.4])
    X_txt_lex_te = np.hstack([B_te_t1, B_te_t2, B_te_t3, B_te_t4, B_te_tfidf * 1.4])
    lr_txt = LogisticRegression(C=0.5, max_iter=2000, random_state=42 + fold)
    lr_txt.fit(X_txt_lex_tr, y_tr)
    p_txt_va = 0.50 * lr_txt.predict_proba(X_txt_lex_va) + 0.30 * oof_ft_hrob[val_idx] + 0.20 * oof_ft_sent[val_idx]
    p_txt_te = 0.50 * lr_txt.predict_proba(X_txt_lex_te) + 0.30 * te_ft_hrob + 0.20 * te_ft_sent
    p_knn_va = 0.5 * cosine_knn_predict(B_tr_all[train_idx], y_tr, B_tr_all[val_idx], k=7, temp=0.06) + 0.5 * cosine_knn_predict(B_tr_all[train_idx], y_tr, B_tr_all[val_idx], k=13, temp=0.08)
    p_knn_te = 0.5 * cosine_knn_predict(B_tr_all[train_idx], y_tr, B_te_all, k=7, temp=0.06) + 0.5 * cosine_knn_predict(B_tr_all[train_idx], y_tr, B_te_all, k=13, temp=0.08)
    B_aud_tr = np.hstack([B_tr_a1[train_idx], B_tr_a2[train_idx], B_tr_p[train_idx], B_tr_pr[train_idx]])
    B_aud_va = np.hstack([B_tr_a1[val_idx], B_tr_a2[val_idx], B_tr_p[val_idx], B_tr_pr[val_idx]])
    B_aud_te = np.hstack([B_te_a1, B_te_a2, B_te_p, B_te_pr])
    svc_aud = SVC(C=2.0, kernel='rbf', probability=True, random_state=42 + fold)
    svc_aud.fit(B_aud_tr, y_tr)
    p_aud_va = svc_aud.predict_proba(B_aud_va)
    p_aud_te = svc_aud.predict_proba(B_aud_te)
    sim_a_tr, _ = compute_class_prototype_sims(B_aud_tr, y_tr, B_aud_tr, B_aud_te)
    sim_a_va, sim_a_te = compute_class_prototype_sims(B_aud_tr, y_tr, B_aud_va, B_aud_te)
    sim_t_tr, _ = compute_class_prototype_sims(X_txt_lex_tr, y_tr, X_txt_lex_tr, X_txt_lex_te)
    sim_t_va, sim_t_te = compute_class_prototype_sims(X_txt_lex_tr, y_tr, X_txt_lex_va, X_txt_lex_te)
    svd = TruncatedSVD(n_components=48, random_state=42 + fold)
    svd_tr = np.hstack([svd.fit_transform(B_tr_all[train_idx]), X_tr_p[train_idx], X_tr_pr[train_idx], sim_a_tr, sim_t_tr])
    svd_va = np.hstack([svd.transform(B_tr_all[val_idx]), X_tr_p[val_idx], X_tr_pr[val_idx], sim_a_va, sim_t_va])
    svd_te = np.hstack([svd.transform(B_te_all), X_te_p, X_te_pr, sim_a_te, sim_t_te])
    clf_lgb = lgb.LGBMClassifier(n_estimators=550, learning_rate=0.025, num_leaves=23, subsample=0.8, colsample_bytree=0.8, random_state=42 + fold, verbosity=-1)
    clf_lgb.fit(svd_tr, y_tr)
    p_lgb_va = clf_lgb.predict_proba(svd_va)
    p_lgb_te = clf_lgb.predict_proba(svd_te)
    clf_cat = CatBoostClassifier(iterations=650, learning_rate=0.03, depth=5, random_seed=42 + fold, verbose=0)
    clf_cat.fit(svd_tr, y_tr)
    p_cat_va = clf_cat.predict_proba(svd_va)
    p_cat_te = clf_cat.predict_proba(svd_te)
    fold_oof = (
        0.15 * p_lrbn_va +
        0.18 * p_svcbn_va +
        0.11 * p_svcstd_va +
        0.09 * p_lrstd_va +
        0.12 * p_mlp_va +
        0.15 * p_gmu_va +
        0.08 * p_txt_va +
        0.04 * p_knn_va +
        0.04 * p_aud_va +
        0.02 * p_lgb_va +
        0.02 * p_cat_va
    )
    fold_te = (
        0.15 * p_lrbn_te +
        0.18 * p_svcbn_te +
        0.11 * p_svcstd_te +
        0.09 * p_lrstd_te +
        0.12 * p_mlp_te +
        0.15 * p_gmu_te +
        0.08 * p_txt_te +
        0.04 * p_knn_te +
        0.04 * p_aud_te +
        0.02 * p_lgb_te +
        0.02 * p_cat_te
    )
    tr_fold_wavs = [train_wavs[i] for i in train_idx]
    va_fold_wavs = [train_wavs[i] for i in val_idx]
    has_va_twin, va_twin_p = find_exact_waveform_twins(va_fold_wavs, tr_fold_wavs, y_tr, exclude_self=False)
    fold_oof[has_va_twin] = 0.94 * va_twin_p[has_va_twin] + 0.06 * fold_oof[has_va_twin]
    oof_probs[val_idx] = fold_oof
    test_probs += fold_te / n_splits
    f1 = f1_score(y_va, np.argmax(fold_oof, axis=1), average='macro')
    print(f"Fold {fold+1} Macro F1: {f1:.4f} (Exact Twins Matched in Val: {has_va_twin.sum()})")
has_te_twin, te_twin_p = find_exact_waveform_twins(test_wavs, train_wavs, y_train, exclude_self=False)
test_probs[has_te_twin] = 0.97 * te_twin_p[has_te_twin] + 0.03 * test_probs[has_te_twin]
for grp in test_twin_groups:
    test_probs[grp] = np.mean(test_probs[grp], axis=0, keepdims=True)
print(f"Stage 1 OOF Macro F1: {f1_score(y_train, np.argmax(oof_probs, axis=1), average='macro'):.4f}")
cur_oof = oof_probs.copy()
cur_te = test_probs.copy()
for s_round, conf_thresh in enumerate([0.84, 0.78]):
    test_conf = np.max(cur_te, axis=1)
    hc_mask = test_conf >= conf_thresh
    pseudo_B = B_te_all[hc_mask]
    pseudo_X = X_te_all[hc_mask]
    pseudo_y = np.argmax(cur_te[hc_mask], axis=1)
    pseudo_w = (test_conf[hc_mask] ** 2) * 0.92
    print(f"Semi-Supervised Round {s_round+1} (thresh={conf_thresh}): {hc_mask.sum()} / {len(test_df)} anchors")
    next_oof = np.zeros_like(cur_oof)
    next_te = np.zeros_like(cur_te)
    for fold, (train_idx, val_idx) in enumerate(skf_splits):
        B_tr_aug = np.vstack([B_tr_all[train_idx], pseudo_B])
        X_tr_aug = np.vstack([X_tr_all[train_idx], pseudo_X])
        y_tr_aug = np.concatenate([y_train[train_idx], pseudo_y])
        w_tr_aug = np.concatenate([np.ones(len(train_idx)), pseudo_w])
        lr_ref = LogisticRegression(C=0.45, max_iter=2500, random_state=200 + s_round*20 + fold)
        lr_ref.fit(B_tr_aug, y_tr_aug, sample_weight=w_tr_aug)
        svc_bn_ref = SVC(C=2.4, kernel='rbf', probability=True, random_state=200 + s_round*20 + fold)
        svc_bn_ref.fit(B_tr_aug, y_tr_aug, sample_weight=w_tr_aug)
        svc_std_ref = SVC(C=2.2, kernel='rbf', probability=True, random_state=200 + s_round*20 + fold)
        svc_std_ref.fit(X_tr_aug, y_tr_aug, sample_weight=w_tr_aug)
        mlp_ref = MLPClassifier(hidden_layer_sizes=(512, 128), activation='relu', alpha=0.12, max_iter=600, random_state=200 + s_round*20 + fold)
        mlp_ref.fit(B_tr_aug, y_tr_aug)
        p_va_ref = (
            0.30 * lr_ref.predict_proba(B_tr_all[val_idx]) +
            0.32 * svc_bn_ref.predict_proba(B_tr_all[val_idx]) +
            0.18 * svc_std_ref.predict_proba(X_tr_all[val_idx]) +
            0.20 * mlp_ref.predict_proba(B_tr_all[val_idx])
        )
        p_te_ref = (
            0.30 * lr_ref.predict_proba(B_te_all) +
            0.32 * svc_bn_ref.predict_proba(B_te_all) +
            0.18 * svc_std_ref.predict_proba(X_te_all) +
            0.20 * mlp_ref.predict_proba(B_te_all)
        )
        next_oof[val_idx] = 0.62 * cur_oof[val_idx] + 0.38 * p_va_ref
        next_te += (0.62 * cur_te + 0.38 * p_te_ref) / n_splits
        tr_fold_wavs = [train_wavs[i] for i in train_idx]
        va_fold_wavs = [train_wavs[i] for i in val_idx]
        has_va_twin, va_twin_p = find_exact_waveform_twins(va_fold_wavs, tr_fold_wavs, y_train[train_idx], exclude_self=False)
        next_oof[val_idx[has_va_twin]] = 0.95 * va_twin_p[has_va_twin] + 0.05 * next_oof[val_idx[has_va_twin]]
    next_te[has_te_twin] = 0.97 * te_twin_p[has_te_twin] + 0.03 * next_te[has_te_twin]
    for grp in test_twin_groups:
        next_te[grp] = np.mean(next_te[grp], axis=0, keepdims=True)
    cur_oof = next_oof
    cur_te = next_te
oof_probs_refined = cur_oof
test_probs_refined = cur_te
def loss_func(weights):
    w = np.exp(weights)
    preds = np.argmax(oof_probs_refined * w, axis=1)
    return -f1_score(y_train, preds, average='macro')
opt = minimize(loss_func, x0=[0.0, 0.0, 0.0, 0.0], method='Nelder-Mead')
calibrated_weights = np.exp(opt.x)
oof_preds_raw = np.argmax(oof_probs_refined, axis=1)
oof_preds_cal = np.argmax(oof_probs_refined * calibrated_weights, axis=1)
raw_f1 = f1_score(y_train, oof_preds_raw, average='macro')
cal_f1 = f1_score(y_train, oof_preds_cal, average='macro')
if cal_f1 <= raw_f1:
    calibrated_weights = np.ones(4)
    oof_preds_cal = oof_preds_raw
    cal_f1 = raw_f1
print(f"\n{'='*20} V9 MONARCH OOF Macro F1 Score: {cal_f1:.4f} {'='*20}\n")
print(classification_report(y_train, oof_preds_cal, target_names=target_emotions))


In [ ]:
final_probs = test_probs_refined * calibrated_weights
for grp in test_twin_groups:
    final_probs[grp] = np.mean(final_probs[grp], axis=0, keepdims=True)
final_preds = np.argmax(final_probs, axis=1)
predicted_emotions = [id2label[p] for p in final_preds]
sub_df = pd.DataFrame({
    'Id': test_df['Id'],
    'emotion': predicted_emotions
})
sub_df.to_csv('submission.csv', index=False)
print("Saved submission.csv")
print(f"Rows: {len(sub_df)}")
solutions_df = pd.DataFrame({
    'AUDIO_ID': [f"{int(x):04d}" for x in test_df['Id']],
    'PREDICTED LABEL': [e.capitalize() for e in predicted_emotions]
}).sort_values('AUDIO_ID')
solutions_df.to_csv('solutions.csv', index=False)
print("Saved solutions.csv")
print("\nTest Set Prediction Distribution:")
print(pd.Series(predicted_emotions).value_counts())
